# Llama-3.1-8B-Instruct full bf16 harvest (Dennis's WS1 task, run for him)

Dennis's script from *Official Copy of Full research* (cell 41) passed its 50-prompt test on 11 Nov
(`llama31-8b__test__hf-bf16-c1__ea9a30bb.tpxz`) but the full run was never started. This notebook runs it.

- Same script, bf16, no NF4, one L40S, MRG sponsor workspace, Modal volume `dennis-data`.
- **One change:** prompts are stripped, de-duplicated and ordered the same way as his Qwen2.5-7B harvest, so
  prompt ids match across the two models (his Llama script hashed unstripped text).
- Expected cost: about 20-30 min of L40S, roughly **$1** of team credits (Qwen-7B took about 20 min).
- Run the cells in order. Step 4 runs a 50-prompt test first; only start step 5 if step 4 looks right.
- Finish with step 7: it checks that no Modal app is left running.

## 1. Secrets, Modal and workspace check

In [ ]:
from google.colab import userdata
import os
for name in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET", "HF_TOKEN"):
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        print("Missing Colab secret:", name)
!pip -q install modal
!python -m modal token info | grep -i workspace
# Must say dylanlerhj (the MRG sponsor workspace). If it shows anything else, stop here.

## 2. Check the Hugging Face token can download Llama 3.1 (gated model)

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(token=os.environ.get("HF_TOKEN"))
print("HF user:", api.whoami()["name"])
try:
    hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json", token=os.environ.get("HF_TOKEN"))
    print("OK: this token can download meta-llama/Llama-3.1-8B-Instruct")
except Exception as e:
    print("NO ACCESS:", type(e).__name__, e)
    print("Request access on the model page, or the harvest will fail at model loading.")

## 3. Check Dennis's volume has the pairs table and see what Llama archives already exist

In [ ]:
!python -m modal volume ls dennis-data /
!python -m modal volume ls dennis-data /llama31_activations

## 4. Write the script and run the 50-prompt test (about 2-4 min)

In [7]:
%%writefile modal_llama31_harvest.py
# Dennis's modal_llama31_harvest.py (Official Copy of Full research, cell 41), with one fix:
# prompts are now read, stripped, de-duplicated and ordered exactly like his Qwen2.5-7B
# harvest (modal_qwen_harvest.py), so prompt_id = sha1(stripped text)[:16] matches the
# Qwen archives and the same pairs join across both models. Everything else is unchanged:
# bf16, no quantisation, L40S, 500-prompt shards, same extraction spec and spec hash.
import os
import time
import shutil
import hashlib
import subprocess
from pathlib import Path
import modal

app = modal.App("dennis-llama31-harvest")
volume = modal.Volume.from_name("dennis-data", create_if_missing=False)

image = (
    modal.Image.debian_slim(python_version="3.12")
    .apt_install("pixz", "tar")
    .pip_install("torch", "transformers", "accelerate", "safetensors",
                 "pandas", "pyarrow", "pyyaml", "huggingface_hub")
)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_TAG = "llama31-8b"
CONFIG_NAME = "hf-bf16-c1"   # Reference Config C1
N_LAYERS = 32
HIDDEN = 4096
SHARD_PROMPTS = 500
MAX_PROMPT_TOKENS = 4096
INST_OFFSET = -6             # last user token, checked in Dennis's preflight (cell 40)

PREFILL_LAYERS = list(range(N_LAYERS))
INST_LAYERS = list(range(1, N_LAYERS, 2))
DECODE_LAYERS = [7, 15, 23, 31]
DECODE_POSITIONS = [0, 4, 9, 19]

POINTS = {
    "last_prefill": dict(layers=PREFILL_LAYERS, position="prompt[-1]"),
    "last_instruction": dict(layers=INST_LAYERS, position=f"prompt[{INST_OFFSET}]"),
}
for _k in DECODE_POSITIONS:
    # Kept so the spec hash matches Dennis's test archive. No generation is run, so these
    # rows stay zero, the same as in his Qwen2.5-7B bf16 archives.
    POINTS[f"gen_{_k}"] = dict(layers=DECODE_LAYERS, position=f"generated[{_k}]")

ROWS = [(p, l) for p, s in POINTS.items() for l in s["layers"]]
ROW_OF = {key: i for i, key in enumerate(ROWS)}


def sha8(obj):
    import json
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:8]


SPEC = {
    "version": "1",
    "extraction_spec": {
        "model": MODEL_ID, "config": CONFIG_NAME, "layers": N_LAYERS,
        "hidden": HIDDEN, "points": POINTS, "dtype": "bfloat16",
    },
}
SPEC_HASH = sha8(SPEC)   # Dennis's test archive: ea9a30bb


def load_prompts():
    """Same steps as modal_qwen_harvest.py, so prompt ids and order match the Qwen run."""
    import pandas as pd
    candidates = [Path("/data/pairs_table.csv"), Path("/data/data/pairs_table.csv")]
    data_path = next((p for p in candidates if p.exists()), None)
    if not data_path:
        raise FileNotFoundError(f"Pairs table not found under /data: {list(Path('/data').glob('*'))}")
    print(f"Reading pairs table from: {data_path}")
    df = pd.read_csv(data_path, dtype=str, keep_default_na=False)
    van_col = next(c for c in ("vanilla_prompt", "vanilla") if c in df.columns)
    adv_col = "adversarial_prompt"
    ds_col = next((c for c in ("dataset", "source") if c in df.columns), None)
    df[van_col] = df[van_col].astype(str).str.strip()
    df[adv_col] = df[adv_col].astype(str).str.strip()
    df = df[(df[van_col] != "") & (df[adv_col] != "")
            & (df[van_col] != "nan") & (df[adv_col] != "nan")].copy()
    df["_pid"] = df["pair_id"].astype(str)
    df["_ds"] = df[ds_col].astype(str) if ds_col else "pairs"
    long = pd.concat([
        df[["_pid", "_ds"]].assign(role="vanilla", text=df[van_col]),
        df[["_pid", "_ds"]].assign(role="adv", text=df[adv_col]),
    ])
    long["prompt_id"] = long["text"].map(lambda t: hashlib.sha1(str(t).encode("utf-8")).hexdigest()[:16])
    return long.drop_duplicates("prompt_id").sort_values(["_ds", "prompt_id"]).reset_index(drop=True)


@app.function(
    image=image,
    gpu="L40S",
    timeout=4 * 3600,   # hard stop; the Qwen-7B run took about 20 minutes
    volumes={"/data": volume},
    secrets=[modal.Secret.from_dict({"HF_TOKEN": os.environ.get("HF_TOKEN", "")})],
    retries=0,          # no retries, so an error cannot keep spending credits
)
def run_harvest(test_only: bool = False):
    os.environ["HF_HUB_DISABLE_XET"] = "1"     # the xet download hung at 0/4 files on 13 Nov; plain HTTP instead
    os.environ["HF_HOME"] = "/data/hf_cache"   # weights kept on the volume, so later jobs skip the 16 GB download
    import torch
    import pandas as pd
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from safetensors.torch import save_file

    print(f"HARVESTING {MODEL_ID} [{CONFIG_NAME}] spec {SPEC_HASH}")
    print(f"HF cache: {os.environ['HF_HOME']}  xet disabled: {os.environ.get('HF_HUB_DISABLE_XET')}")
    out_dir = Path("/data/llama31_activations")
    out_dir.mkdir(parents=True, exist_ok=True)

    uniq = load_prompts()
    if test_only:
        uniq = uniq.head(50)
    print(f"Prompts to process: {len(uniq):,}")

    hf_token = os.environ.get("HF_TOKEN")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda", token=hf_token).eval()
    assert next(model.parameters()).dtype == torch.bfloat16
    volume.commit()   # keep the downloaded weights
    print(f"Model loaded in bf16. VRAM {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

    buf = torch.zeros((len(ROWS), HIDDEN), dtype=torch.bfloat16, device="cuda")

    def make_hook(layer):
        pre_row = ROW_OF.get(("last_prefill", layer))
        inst_row = ROW_OF.get(("last_instruction", layer))
        def hook(module, args, output):
            hs = output[0] if isinstance(output, tuple) else output
            if pre_row is not None:
                buf[pre_row] = hs[0, -1]
            if inst_row is not None:
                buf[inst_row] = hs[0, INST_OFFSET]
        return hook

    for i, layer_mod in enumerate(model.model.layers):
        layer_mod.register_forward_hook(make_hook(i))

    size = 50 if test_only else SHARD_PROMPTS
    chunks = [uniq.iloc[i:i + size] for i in range(0, len(uniq), size)]
    print(f"{len(chunks)} shards")

    for si, chunk in enumerate(chunks):
        tag = f"test_{si:05d}" if test_only else f"s{si:05d}"
        dest = out_dir / f"{MODEL_TAG}__{tag}__{CONFIG_NAME}__{SPEC_HASH}.tpxz"
        if dest.exists() and not test_only:
            print(f"Skipping {dest.name} (already on volume)")
            continue
        t0 = time.time()
        bufs, index = [], []
        for _, row in chunk.iterrows():
            rendered = tokenizer.apply_chat_template(
                [{"role": "user", "content": row["text"]}], tokenize=False, add_generation_prompt=True)
            enc = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
            n_tok = enc["input_ids"].shape[1]
            if n_tok > MAX_PROMPT_TOKENS:
                bufs.append(torch.zeros((len(ROWS), HIDDEN), dtype=torch.bfloat16))
                index.append({"prompt_id": row["prompt_id"], "token_count": n_tok, "skipped": True})
                continue
            buf.zero_()
            with torch.no_grad():
                model(enc["input_ids"].to("cuda"))
            bufs.append(buf.cpu().clone())
            index.append({"prompt_id": row["prompt_id"], "token_count": n_tok, "skipped": False})

        tmp = Path(f"/tmp/shard_{tag}")
        tmp.mkdir(parents=True, exist_ok=True)
        arr = torch.stack(bufs)
        tensors = {f"{p}.layer_{l}": arr[:, ROW_OF[(p, l)], :].contiguous() for (p, l) in ROWS}
        save_file(tensors, str(tmp / "activations.safetensors"))
        pd.DataFrame(index).to_parquet(tmp / "index.parquet", index=False)
        subprocess.run(f"tar -C {tmp.parent} --use-compress-program='pixz -1' -cf '{dest}' {tmp.name}",
                       shell=True, check=True)
        shutil.rmtree(tmp)
        del arr, tensors, bufs
        torch.cuda.empty_cache()
        volume.commit()
        print(f"Archived {dest.name} ({len(chunk)} prompts) in {time.time() - t0:.1f}s")

    print("Llama-3.1-8B bf16 harvest finished.")


@app.local_entrypoint()
def main(test: bool = False):
    run_harvest.remote(test_only=test)

Overwriting modal_llama31_harvest.py


In [8]:
!python -m modal run modal_llama31_harvest.py --test
# Expect: Prompts to process: 50, Model loaded in bf16, Archived llama31-8b__test_00000__hf-bf16-c1__ea9a30bb.tpxz

✓ Initialized. View run at 
https://modal.com/apps/dylanlerhj/main/ap-XNezqB7LxUosW8HdJljXRK
⠋ Initializing...
⠸ Creating objects...
└── ⠋ Creating mount /content/modal_llama31_harvest.py: Uploaded 0/1 missing 
⠦ Creating objects...
└── ⠸ Creating mount /content/modal_llama31_harvest.py: Uploaded 0/1 missing 
⠏ Creating objects...
└── ⠦ Creating mount /content/modal_llama31_harvest.py: Finalizing index of 1 
⠹ Creating objects...
├── ✓ Uploaded /content/modal_llama31_harvest.py
└── ✓ Created Function run_harvest.
✓ Created objects.
├── ✓ Uploaded /content/modal_llama31_harvest.py
└── ✓ Created Function run_harvest.
⠹ Running... View app at 
⠴ Loading images (1 containers initializing)... View app at 
⠇ Loading images (1 containers initializing)... View app at 
⠙ Loading images (1 containers initializing)... View app at 
⠼ Loading images (1 containers initializing)... View app at 
⠧ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)..

## 5. Full harvest (detached, so it keeps running if this tab closes)

In [ ]:
!python -m modal run --detach modal_llama31_harvest.py

Streaming output truncated to the last 5000 lines.
⠙ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 

## 6. Progress: re-run this cell to see how many shards are done

In [ ]:
import subprocess, re
out = subprocess.run(["python", "-m", "modal", "volume", "ls", "dennis-data", "/llama31_activations"],
                     capture_output=True, text=True).stdout
done = len(re.findall(r"llama31-8b__s\d{5}", out))
print(f"Full-run shards on volume: {done}")
!python -m modal app list

## 7. When it has finished: confirm nothing is still running

In [ ]:
!python -m modal app list
# Every app should show stopped. If dennis-llama31-harvest is still running after it finished, stop it:
# !python -m modal app stop dennis-llama31-harvest